# 1. Serveur MCP avec templates de prompts

On ajoute une méthode pour exposer des templates de prompts sous forme d'outils ou de ressources. Chaque template est un prompt structuré que l'agent peut utiliser pour interagir avec le LLM.

Fichier : mcp_server_with_prompt_templates.py

## Explications :

### PROMPT_TEMPLATES :
Dictionnaire contenant des templates de prompts pour des scénarios spécifiques (calcul, lecture de fichier, résumé de PDF).

Chaque template contient :

Un nom (ex: calculate_prompt).

Une description (pour le LLM/agent).

Un template (avec des placeholders comme {expression}, {filename}).

list_prompt_templates :

Outil qui liste les templates disponibles (avec leurs placeholders).

get_prompt_template :

Outil qui retourne un template spécifique par son nom.



# 2. Client MCP utilisant les templates de prompts

Le client va :

- Récupérer la liste des templates depuis le serveur.
- Choisir un template en fonction de la requête utilisateur.
- Remplir les placeholders (ex: {expression} → "2 + 2").
- Envoyer le prompt final au LLM (simulé ici).
- Exécuter les actions suggérées par le LLM.

Fichier : mcp_client_with_prompt_templates.py

## Explications :

list_prompt_templates :
- Récupère la liste des templates disponibles (avec leurs placeholders).
get_prompt_template :
- Récupère un template spécifique et le remplit avec les valeurs utilisateur (ex: {expression} → "2 + 2").
ask_llm :
- Simule un appel à un LLM avec le prompt rempli. En pratique, tu utiliserais une API LLM réelle.

Exécution :
Le client parse la réponse du LLM et appelle l'outil correspondant.

### 3. Sortie attendue

Templates de prompts disponibles :
- calculate_prompt : Template pour demander un calcul mathématique. Utilise l'outil `calculate`.
  Placeholders : ['expression']
- read_file_prompt : Template pour demander la lecture d'un fichier. Utilise l'outil `read_file`.
  Placeholders : ['filename']
- summarize_pdf_prompt : Template pour résumer un PDF. Utilise `read_file` puis un LLM pour générer un résumé.
  Placeholders : ['filename']

--- Utilisation du template 'calculate_prompt' ---
Template :
        L'utilisateur demande de calculer l'expression suivante : {expression}.
        Utilise l'outil `calculate(expression: str)` pour obtenir le résultat.
        Retourne uniquement le résultat du calcul.

Prompt rempli :
        L'utilisateur demande de calculer l'expression suivante : 2 + 3 * 4.
        Utilise l'outil `calculate(expression: str)` pour obtenir le résultat.
        Retourne uniquement le résultat du calcul.

Réponse du LLM : {"action": "calculate", "args": {"expression": "2 + 3 * 4"}}
Résultat de calculate : 14

--- Utilisation du template 'read_file_prompt' ---
Template :
        L'utilisateur demande de lire le fichier suivant : {filename}.
        Utilise l'outil `read_file(filename: str)` pour obtenir le contenu du fichier.
        Si le fichier n'existe pas, retourne une erreur claire.

Prompt rempli :
        L'utilisateur demande de lire le fichier suivant : maths.pdf.
        Utilise l'outil `read_file(filename: str)` pour obtenir le contenu du fichier.
        Si le fichier n'existe pas, retourne une erreur claire.

Réponse du LLM : {"action": "read_file", "args": {"filename": "maths.pdf"}}
Contenu du fichier (premiers 50 octets) : b'%PDF-1.4\n%\xE2\xE3\xCF\xD3\n1 0 obj\n<'




## 4. Intégration avec un agent LLM réel
Pour une intégration réelle, voici comment un agent pourrait utiliser ces templates :
Exemple d'agent


### Sortie attendue
```
Résultat : 4
Contenu du fichier (premiers 50 octets) : b'%PDF-1.4\n%\xE2\xE3\xCF\xD3\n1 0 obj\n<'
```



## 5. Avantages de cette approche
    
| Concept | Avantages |
| --- | --- |
| Templates centralisés | Les prompts sont définis côté serveur, ce qui permet une maintenance centralisée (pas besoin de les dupliquer dans chaque client/agent). |
| Réutilisabilité | Un même template peut être utilisé par plusieurs agents ou clients. |
| Flexibilité | Les templates peuvent être modifiés dynamiquement sans changer le code client. |
| Guidage du LLM | Les templates structurent les requêtes au LLM, ce qui améliore la précision et la reproductibilité. |
| Sécurité | Les placeholders permettent de contrôler les entrées (ex: valider filename avant utilisation). |




## 6. Cas d'usage avancés
**a. Templates avec logique conditionnelle**


Tu peux ajouter des règles dans les templates pour guider le LLM :
```
PROMPT_TEMPLATES = {
    "analyze_pdf": {
        "name": "analyze_pdf_prompt",
        "description": "Analyse un PDF et répond à une question.",
        "template": """
        L'utilisateur pose la question suivante sur le fichier {filename} : {question}.
        1. Utilise `read_file(filename: str)` pour lire le PDF.
        2. Réponds à la question en te basant sur le contenu du PDF.
        3. Si la réponse n'est pas dans le PDF, dis-le clairement.
        """,
    },
}
```


**b. Templates avec exemples (few-shot)**

Ajoute des exemples dans les templates pour guider le LLM :
```
"template": """
Exemples :
- Question : "Quelle est la somme de 2 et 2 ?" → Action : calculate("2 + 2")
- Question : "Donne-moi le PDF sur les maths." → Action : read_file("maths.pdf")

Maintenant, traite la requête suivante : {user_query}.
""",
```


**c. Templates dynamiques**

Les templates peuvent être générés dynamiquement en fonction du contexte (ex: langue de l'utilisateur, type de fichier, etc.).

## 7. Résumé des étapes clés

Serveur MCP :

Expose des outils (read_file, calculate).
Expose des templates de prompts (list_prompt_templates, get_prompt_template).

Client/Agent :

Récupère les templates depuis le serveur.
Sélectionne le template pertinent en fonction de la requête utilisateur.
Remplit les placeholders (ex: {filename} → "maths.pdf").
Envoie le prompt rempli au LLM.
Exécute les actions suggérées par le LLM.

LLM :

Reçoit un prompt structuré (grâce au template).
Suggère une action (ex: {"action": "read_file", "args": {"filename": "maths.pdf"}}).


Pourquoi cette architecture ?

Découplage : Le serveur gère les outils et les templates, l'agent gère la logique métier.
Évolutivité : Ajouter un nouvel outil ou template ne nécessite pas de modifier les clients.
Contrôle : Les templates permettent de standardiser les interactions avec le LLM.
